# 🧪 Test Scripts
Notebook tổng hợp các scripts test cho project NutriTrack.

---

### ⚙️ 0. Setup Environment
**Bắt buộc chạy cell này đầu tiên** để thiết lập đường dẫn và load API keys.

In [1]:
import sys
import os
from dotenv import load_dotenv

# 1. Thêm project root vào sys.path để có thể import module 'app'
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"✅ Added to path: {project_root}")

# 2. Load API keys từ .env
env_path = os.path.join(project_root, "app", "config", ".env")
if os.path.exists(env_path):
    load_dotenv(env_path)
    USDA_API_KEY = os.getenv("USDA_API_KEY")
    print(f"✅ .env loaded from: {env_path}")
    print(f"✅ USDA API Key: {USDA_API_KEY[:8]}..." if USDA_API_KEY else "❌ USDA_API_KEY not found in .env")
else:
    print(f"❌ .env file not found at: {env_path}")

✅ Added to path: d:\Project\Code\nutritrack-documentation
✅ .env loaded from: d:\Project\Code\nutritrack-documentation\app\config\.env
✅ USDA API Key: pLy0emrT...


## 🍣 Test 1: USDA Nutrition Retrieval
Kiểm tra khả năng kết nối và lấy dữ liệu từ USDA API.

In [2]:
from app.third_apis.USAD import USDAClient

# Initialize client
client = USDAClient(api_key=os.getenv("USDA_API_KEY"))

ingredients = ["Rice", "Salmon", "Tuna", "Shrimp", "Octopus"]
for ing in ingredients:
    nutrition = client.get_nutrition(ing)
    print(f"✅ {ing:<12} -> {nutrition}")


{'fdcId': 2077766, 'description': 'RICE', 'dataType': 'Branded', 'gtinUpc': '817367010166', 'publishedDate': '2021-10-28', 'brandOwner': 'Windmill Rice Co., LLC', 'brandName': 'WINDMILL RICE', 'ingredients': 'LONG GRAIN RICE ENRICHED WITH IRON (FERRIC ORTHOPHOSPHATE), NIACIN, THIAMINE (THIAMINE MONONITRATE) AND FOLIC ACID.', 'marketCountry': 'United States', 'foodCategory': 'Rice', 'modifiedDate': '2017-07-14', 'dataSource': 'LI', 'packageWeight': '20 lbs/9.07 kg', 'servingSizeUnit': 'g', 'servingSize': 45.0, 'householdServingFullText': '0.25 cup', 'tradeChannels': ['NO_TRADE_CHANNEL'], 'allHighlightFields': '<b>Brand Owner</b>: Windmill <em>Rice</em> Co., LLC<br /><b>Ingredients</b>: LONG GRAIN <em>RICE</em> ENRICHED WITH IRON (FERRIC ORTHOPHOSPHATE), NIACIN, THIAMINE (THIAMINE MONONITRATE) AND FOLIC ACID.', 'score': 996.3319, 'microbes': [], 'foodNutrients': [{'nutrientId': 1003, 'nutrientName': 'Protein', 'nutrientNumber': '203', 'unitName': 'G', 'derivationCode': 'LCCS', 'derivatio

## 📸 Test 2: Pipeline - Qwen3VL analyze_food ➡️ USDA Nutrition Calculation

**Mục tiêu:**
1. Dùng `Qwen3VL.analyze_food()` phát hiện các món ăn và ingredients.
2. **Làm gọn logic:** Tính toán và lưu kết quả vào biến `final_nutrition_results` để tái sử dụng làm API.
3. **Tách biệt hiển thị:** Dùng cell riêng để render kết quả.

In [ ]:
from app.models.QWEN3VL import Qwen3VL
import os

# Initialize Qwen3VL
qwen = Qwen3VL()

In [ ]:
# Path ảnh test
image_path = os.path.join(project_root, "data", "images", "food", "com_tam.jpg")

print(f"📸 Phân tích ảnh: {image_path}")
food_list = qwen.analyze_food(image_path)
print(f"\n✅ Phát hiện {len(food_list.items)} món ăn.")

In [ ]:
food_list.model_dump()

In [29]:
import time

# ⚙️ CALCULATION CELL: Xử lý logic và lưu vào biến 'final_nutrition_results'
# Biến này có thể dùng làm Response cho API sau này
final_nutrition_results = []

print("🔄 Đang tính toán dinh dưỡng thực tế từ USDA...")

for food in food_list.items:
    food_data = {
        "name": food.name,
        "vi_name": food.vi_name,
        "qwen_estimated_calories": food.total_estimated_calories,
        "ingredients": []
    }
    
    total_nutrition = {"calories": 0, "protein": 0, "carbs": 0, "fat": 0}
    
    for ing in food.ingredients:
        # 1. Tra cứu USDA (luôn trả về per 100g)
        usda_100g = client.get_nutrition(ing.name)
        
        # 2. Tính toán dựa trên trọng lượng thực
        weight = ing.estimated_weight_g if ing.estimated_weight_g else 0
        ratio = weight / 100.0
        
        nutrition = {
            "calories": usda_100g['calories'] * ratio,
            "protein": usda_100g['protein'] * ratio,
            "carbs": usda_100g['carbs'] * ratio,
            "fat": usda_100g['fat'] * ratio
        }
        
        # 3. Lưu thông tin ingredient
        food_data["ingredients"].append({
            "name": ing.name,
            "vi_name": ing.vi_name,
            "weight_g": weight,
            "usda_100g": usda_100g,
            "nutrition": nutrition
        })
        
        # Cộng dồn tổng
        for k in total_nutrition: total_nutrition[k] += nutrition[k]
        
        time.sleep(0.2) # Rate limit
    
    food_data["total_nutrition"] = total_nutrition
    avg_calories = (total_nutrition["calories"] + food.total_estimated_calories) / 2
    food_data["average_calories"] = avg_calories
    final_nutrition_results.append(food_data)

print("✅ Tính toán xong. Dữ liệu đã sẵn sàng trong biến 'final_nutrition_results'.")

🔄 Đang tính toán dinh dưỡng thực tế từ USDA...
✅ Tính toán xong. Dữ liệu đã sẵn sàng trong biến 'final_nutrition_results'.


In [30]:
final_nutrition_results

[{'name': 'Fried Potatoes',
  'vi_name': 'Khoai tây chiên',
  'qwen_estimated_calories': 200.0,
  'ingredients': [{'name': 'Potatoes',
    'vi_name': 'Khoai tây',
    'weight_g': 150.0,
    'usda_100g': {'calories': 74.0, 'protein': 2.7, 'fat': 0.0, 'carbs': 16.9},
    'nutrition': {'calories': 111.0,
     'protein': 4.050000000000001,
     'carbs': 25.349999999999998,
     'fat': 0.0}}],
  'total_nutrition': {'calories': 111.0,
   'protein': 4.050000000000001,
   'carbs': 25.349999999999998,
   'fat': 0.0},
  'average_calories': 155.5},
 {'name': 'Fried Chicken Sandwich',
  'vi_name': 'Bánh mì gà rán',
  'qwen_estimated_calories': 450.0,
  'ingredients': [{'name': 'Chicken',
    'vi_name': 'Gà',
    'weight_g': 120.0,
    'usda_100g': {'calories': 188,
     'protein': 24.7,
     'fat': 9.41,
     'carbs': 1.18},
    'nutrition': {'calories': 225.6,
     'protein': 29.639999999999997,
     'carbs': 1.416,
     'fat': 11.292}},
   {'name': 'Bun',
    'vi_name': 'Bánh mì',
    'weight_g'

In [31]:
# 📊 DISPLAY CELL: Hiển thị báo cáo từ 'final_nutrition_results'
print(f"{'='*95}")
print(f"📊 BÁO CÁO DINH DƯỠNG CHI TIẾT")
print(f"{'='*95}\n")

for food in final_nutrition_results:
    print(f"🍽️  MÓN: {food['name']} ({food['vi_name']})")
    print(f"🔥 Calories (AI ước tính): {food['qwen_estimated_calories']} kcal")
    print(f"{'Ingredient':<20} | {'Weight':>8} | {'USDA(100g) Cal/P/C/F':<24} | {'REAL Cal/P/C/F':<24}")
    print("-" * 95)
    
    for ing in food['ingredients']:
        u = ing['usda_100g']
        r = ing['nutrition']
        
        u_str = f"{u['calories']:>5.0f}/{u['protein']:>4.1f}/{u['carbs']:>4.1f}/{u['fat']:>4.1f}"
        r_str = f"{r['calories']:>5.0f}/{r['protein']:>4.1f}/{r['carbs']:>4.1f}/{r['fat']:>4.1f}"
        
        print(f"   {ing['name']:<17} | {ing['weight_g']:>7.1f}g | {u_str:<24} | {r_str:<24}")
    
    t = food['total_nutrition']
    avg_calories = food["average_calories"]
    print("-" * 95)
    print(f"📊 TỔNG CỘNG THỰC TẾ: {'':<10} | Avg Calories: {avg_calories:>4.1f}g | Cal: {t['calories']:>5.0f} | Pro: {t['protein']:>4.1f}g | Carb: {t['carbs']:>4.1f}g | Fat: {t['fat']:>4.1f}g")
    print("\n" + "="*95 + "\n")

📊 BÁO CÁO DINH DƯỠNG CHI TIẾT

🍽️  MÓN: Fried Potatoes (Khoai tây chiên)
🔥 Calories (AI ước tính): 200.0 kcal
Ingredient           |   Weight | USDA(100g) Cal/P/C/F     | REAL Cal/P/C/F          
-----------------------------------------------------------------------------------------------
   Potatoes          |   150.0g |    74/ 2.7/16.9/ 0.0     |   111/ 4.1/25.3/ 0.0    
-----------------------------------------------------------------------------------------------
📊 TỔNG CỘNG THỰC TẾ:            | Avg Calories: 155.5g | Cal:   111 | Pro:  4.1g | Carb: 25.3g | Fat:  0.0g


🍽️  MÓN: Fried Chicken Sandwich (Bánh mì gà rán)
🔥 Calories (AI ước tính): 450.0 kcal
Ingredient           |   Weight | USDA(100g) Cal/P/C/F     | REAL Cal/P/C/F          
-----------------------------------------------------------------------------------------------
   Chicken           |   120.0g |   188/24.7/ 1.2/ 9.4     |   226/29.6/ 1.4/11.3    
   Bun               |    80.0g |   273/11.2/39.8/ 7.6     |  